In [1]:
import pandas as pd 
import numpy as np 
import seaborn as sns 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error,root_mean_squared_error


import sys 
from pathlib import Path 
sys.path.append(str(Path().resolve().parent))

from App_files.dataframe_loader import df_loader
from App_files.feature_engineering import feature_engineered
from App_files.data_loader import target_risk, train_test_split



In [2]:
df = df_loader()

2025-07-29 21:50:41 | INFO | root | Creating Connection to the database
2025-07-29 21:50:41 | INFO | root | Connection created successfully
2025-07-29 21:50:41 | INFO | root | Creating tables
2025-07-29 21:50:41 | INFO | root | Tables created successfully
2025-07-29 21:50:41 | INFO | root | Loading the database and Normalizing for a 3NF..
2025-07-29 21:50:41 | INFO | root | Normalization done !.
2025-07-29 21:50:41 | INFO | root | Inserting data to the tables
2025-07-29 21:50:41 | INFO | root | All data inserted successfully


Removed exisiting database /Users/ajit/Desktop/loan_predictor_app/data/LoanDATABASE.db
Connected to database: /Users/ajit/Desktop/loan_predictor_app/data/LoanDATABASE.db
Table created successfully.
Table created successfully.
Table created successfully.
Table created successfully.
Table created successfully.
Data inserted successfully.
Data inserted successfully.
Data inserted successfully.
Data inserted successfully.
Data inserted successfully.


2025-07-29 21:50:41 | INFO | root | 🧹 Tables created, data inserted, and DB connection closed.


In [3]:
df = feature_engineered(df)

In [4]:
df.columns

Index(['ApplicantID', 'ApplicationDate', 'Age', 'MaritalStatus',
       'NumberOfDependents', 'HomeOwnershipStatus', 'AnnualIncome',
       'MonthlyIncome', 'SavingsAccountBalance', 'CheckingAccountBalance',
       'TotalAssets', 'TotalLiabilities', 'NetWorth', 'DebtToIncomeRatio',
       'TotalDebtToIncomeRatio', 'CreditScore', 'PaymentHistory',
       'NumberOfOpenCreditLines', 'NumberOfCreditInquiries',
       'CreditCardUtilizationRate', 'BankruptcyHistory',
       'PreviousLoanDefaults', 'UtilityBillsPaymentHistory',
       'LengthOfCreditHistory', 'EmploymentStatus', 'EducationLevel',
       'Experience', 'JobTenure', 'LoanAmount', 'LoanDuration', 'LoanPurpose',
       'MonthlyLoanPayment', 'BaseInterestRate', 'InterestRate',
       'LoanApproved', 'RiskScore', 'IncomePerDependent',
       'AssetToLiabilityRatio', 'LoanToIncomeRatio',
       'CreditHistoryLengthPerAge'],
      dtype='object')

In [ ]:
drop_irrelevants = ["ApplicantID","ApplicationDate","HomeOwnershipStatus","Age"]
drop_leakage = ["LoanApproved","TotalDebtToIncomeRatio","CreditScore",'InterestRate', 'BaselInterestRate',]

df.drop(columns=drop_irrelevants, errors="ignore")


,MaritalStatus,NumberOfDependents,AnnualIncome,MonthlyIncome,SavingsAccountBalance,CheckingAccountBalance,TotalAssets,TotalLiabilities,NetWorth,DebtToIncomeRatio,...,LoanPurpose,MonthlyLoanPayment,BaseInterestRate,InterestRate,LoanApproved,RiskScore,IncomePerDependent,AssetToLiabilityRatio,LoanToIncomeRatio,CreditHistoryLengthPerAge
0,Married,2,39948,3329.000000,7632,1202,146111,19183,126928,0.358336,...,Home,419.805992,0.199652,0.227590,0,49.0,1109.666667,7.616295,0.329220,0.195652
1,Single,1,39709,3309.083333,4627,3460,53204,9595,43609,0.330274,...,Debt Consolidation,794.054238,0.207045,0.201077,0,52.0,1654.541667,5.544393,0.655880,0.230769
2,Married,2,40724,3393.666667,886,895,25176,128874,5205,0.244729,...,Education,666.406688,0.217627,0.212548,0,52.0,1131.222222,0.195352,0.432830,0.458333
3,Single,1,69084,5757.000000,1675,1217,104822,5370,99452,0.436244,...,Home,1047.506980,0.300398,0.300911,0,54.0,2878.500000,19.516291,0.548571,0.169492
4,Married,1,103264,8605.333333,1555,4981,244305,17286,227019,0.078884,...,Debt Consolidation,330.179140,0.197184,0.175990,1,36.0,4302.666667,14.132296,0.088936,0.710526
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,Married,3,30180,2515.000000,235,3429,80969,25642,55327,0.468077,...,Auto,905.767712,0.216021,0.195574,0,55.0,628.750000,3.157548,0.812465,0.155556
19996,Married,5,49246,4103.833333,6910,183,69571,5569,64002,0.317372,...,Debt Consolidation,958.395633,0.227318,0.199168,0,54.0,683.972222,12.490305,0.524255,0.491228
19997,Married,3,48958,4079.833333,2175,746,108316,4653,103663,0.023014,...,Home,945.427454,0.229533,0.226766,0,45.0,1019.958333,23.273743,0.756408,0.177778
19998,Married,3,41025,3418.750000,3037,260,22085,11485,10600,0.534517,...,Debt Consolidation,411.168284,0.249760,0.264873,0,59.0,854.687500,1.922776,0.359772,0.213115


In [ ]:
df.drop(columns=drop_leakage, errors="ignore",)

In [ ]:
X,y = target_risk(df)
print(y)

In [ ]:
X_Train,X_Test,y_Train,y_Test = train_test_split(X,y,random_state=42,test_size=0.2)

In [ ]:
print(df["CreditScore"])

In [ ]:
print(X_Train.dtypes[X_Train.dtypes == 'object'])


In [ ]:
# 1. Convert ApplicationDate to datetime and extract year/month
X_Train["ApplicationDate"] = pd.to_datetime(X_Train["ApplicationDate"], errors='coerce')
X_Train["AppYear"] = X_Train["ApplicationDate"].dt.year
X_Train["AppMonth"] = X_Train["ApplicationDate"].dt.month
X_Train = X_Train.drop(columns=["ApplicationDate"])

X_Test["ApplicationDate"] = pd.to_datetime(X_Test["ApplicationDate"], errors='coerce')
X_Test["AppYear"] = X_Test["ApplicationDate"].dt.year
X_Test["AppMonth"] = X_Test["ApplicationDate"].dt.month
X_Test = X_Test.drop(columns=["ApplicationDate"])

# 2. One-hot encode categorical columns
categorical_cols = [
    "MaritalStatus", "HomeOwnershipStatus", "EmploymentStatus",
    "EducationLevel", "LoanPurpose"
]

X_Train = pd.get_dummies(X_Train, columns=categorical_cols)
X_Test = pd.get_dummies(X_Test, columns=categorical_cols)

# 3. Align test columns with train (important after one-hot encoding)
X_Train, X_Test = X_Train.align(X_Test, join='left', axis=1, fill_value=0)


In [ ]:
model = RandomForestRegressor() 
model.fit(X_Train,y_Train)

In [ ]:
y_pred = model.predict(X_Test)

In [ ]:
mse = mean_squared_error(y_Test, y_pred)
print(mse)

In [ ]:
r2 = r2_score(y_Test,y_pred)
print(r2)

In [ ]:
rmse = root_mean_squared_error(y_Test,y_pred)
print(rmse)

In [ ]:
features = pd.Series(model.feature_importances_, index=X_Train.columns)
features.sort_values().plot(kind="barh", title="Feature Importance", figsize=(12,12))